### MLFlow Configuration

In [1]:
import os
import mlflow

TRACKING_URI = os.getenv(
    "MLFLOW_TRACKING_URI",
    "MLFLOW_TRACKING_URI_LOCAL",
    # mlflow.set_tracking_uri("http://127.0.0.1:5000")
)

mlflow.set_tracking_uri(TRACKING_URI)


# Set or create an experiment
# mlflow.set_experiment("Final Experiment using HyperParameter Tuning")

d:\MyFiles\GitHub\YouTube Viewer Sentiment Analysis\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# from mlflow.tracking import MlflowClient

# client = MlflowClient()
# experiments = client.search_experiments(view_type=mlflow.entities.ViewType.DELETED_ONLY)

# for exp in experiments:
#     if exp.name == "Final Experiment using HyperParameter Tuning":
#         client.delete_experiment(exp.experiment_id)  # permanent
#         print("Permanently deleted")

In [5]:
from mlflow.tracking import MlflowClient
import mlflow

client = MlflowClient()

experiments = client.search_experiments(
    view_type=mlflow.entities.ViewType.ALL
)

for exp in experiments:
    print(exp.experiment_id, exp.name, exp.lifecycle_stage)

11 Final Experiment using HyperParameter Tuning active
10 Imbalance_Experiments active
9 Experiment of Handling Imbalanced Data deleted
8 Exp 4 - Handling Imbalanced Data deleted
7 Sentiment_Classification_Max_Features active
6 NLP_Experiments deleted
5 Sentiment_Classification active
0 Default active


In [ ]:
# client.restore_experiment("11")

In [6]:
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns

## Load Cleaned Data

In [7]:
import os
import pandas as pd

# Get the path to the root directory (one level up from this notebook)
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
data_path = os.path.join(ROOT_DIR, 'data', 'processed', 'cleaned_data.csv')

# 1. Load the data

df = pd.read_csv(data_path)

# 2. Map categories (-1 -> 2, 1 -> 1, 0 -> 0)
df['category'] = df['category'].map({-1: 2, 1: 1, 0: 0})

# 3. Check results
df.head()

,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,2
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1


In [8]:
df.isnull().sum()

clean_comment    0
category         0
dtype: int64

## Modeling

### 1. Imbalance Factory

In [9]:
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN


class ImbalanceHandlerFactory:

    @staticmethod
    def get(name: str, random_state=42):

        name = name.lower()

        handlers = {
            "oversampling": SMOTE(random_state=random_state),
            "adasyn": ADASYN(random_state=random_state),
            "undersampling": RandomUnderSampler(random_state=random_state),
            "smote_enn": SMOTEENN(random_state=random_state),
        }

        return handlers.get(name)

### 2. Vectorizer Factory

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer


class VectorizerFactory:
    @staticmethod
    def get(name: str, ngram_range=(1, 1), max_features=8000):
        name = name.lower()

        if name == "tfidf":
            return TfidfVectorizer(
                ngram_range=ngram_range,
                max_features=max_features
            )

        elif name == "count":
            return CountVectorizer(
                ngram_range=ngram_range,
                max_features=max_features
            )

        else:
            raise ValueError(f"Unknown vectorizer: {name}")

###  3. Hyperparameter grids / distributions per model

In [11]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import (
    GridSearchCV, RandomizedSearchCV,
    KFold, StratifiedKFold, cross_val_score
)
import optuna
import numpy as np
optuna.logging.set_verbosity(optuna.logging.WARNING)


# ─────────────────────────────────────────────────────────
# Hyperparameter grids / distributions per model
# ─────────────────────────────────────────────────────────
PARAM_GRIDS = {
    "random_forest": {
        "n_estimators":  [100, 300, 500],
        "max_depth":     [5, 10, 15, None],
        "min_samples_split": [2, 5, 10],
        "max_features":  ["sqrt", "log2"],
    },
    "xgboost": {
        "n_estimators":    [100, 200, 300],
        "learning_rate":   [0.01, 0.05, 0.1],
        "max_depth":       [3, 5, 7],
        "subsample":       [0.7, 0.8, 1.0],
        "colsample_bytree":[0.7, 0.8, 1.0],
    },
    "lightgbm": {
        "n_estimators":  [100, 200, 300],
        "learning_rate": [0.01, 0.05, 0.1],
        "num_leaves":    [31, 63, 127],
        "max_depth":     [-1, 5, 10],
    },
}


def _suggest_params(trial, model_name):
    """Optuna param sampler — called inside the objective function."""
    if model_name == "random_forest":
        return dict(
            n_estimators      = trial.suggest_int  ("n_estimators",    50,  800),
            max_depth         = trial.suggest_int  ("max_depth",        3,   30),
            min_samples_split = trial.suggest_int  ("min_samples_split",2,   20),
            max_features      = trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        )
    elif model_name == "xgboost":
        return dict(
            n_estimators      = trial.suggest_int   ("n_estimators",  50, 400),
            learning_rate     = trial.suggest_float ("learning_rate", 0.005, 0.3, log=True),
            max_depth         = trial.suggest_int   ("max_depth",     2,  10),
            subsample         = trial.suggest_float ("subsample",     0.5, 1.0),
            colsample_bytree  = trial.suggest_float ("colsample_bytree", 0.5, 1.0),
        )
    elif model_name == "lightgbm":
        return dict(
            n_estimators  = trial.suggest_int  ("n_estimators",  50, 400),
            learning_rate = trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
            num_leaves    = trial.suggest_int  ("num_leaves",    20, 200),
            max_depth     = trial.suggest_int  ("max_depth",    -1,  15),
        )
    else:
        raise ValueError(f"No Optuna sampler defined for model: {model_name}")


def _build_base(name, random_state, **params):
    """Instantiate a model by name with the given params."""
    name = name.lower()
    shared = dict(random_state=random_state, **params)
    if name == "random_forest":
        return RandomForestClassifier(**shared)
    elif name == "xgboost":
        return XGBClassifier(eval_metric="logloss", **shared)
    elif name == "lightgbm":
        return LGBMClassifier(verbose=-1, **shared)
    else:
        raise ValueError(f"Unknown model: {name}")


### 4. Model Factory

In [12]:
class ModelFactory:
    """
    Factory that returns a (possibly tuned) estimator.

    search_strategy options
    -----------------------
    'manual'    – fixed hand-crafted hyperparameters (default, no CV needed)
    'grid'      – exhaustive GridSearchCV
    'random'    – RandomizedSearchCV  (n_iter random combos)
    'bayesian'  – Optuna TPE optimisation (n_trials Bayesian trials)

    cv_strategy options
    -------------------
    'train_test'       – simple hold-out (fast but high variance)
    'kfold'            – standard K-Fold
    'stratified_kfold' – Stratified K-Fold (preserves class distribution)
    """

    # ── Manual baseline params ─────────────────────────────
    _MANUAL_PARAMS = {
        "random_forest": dict(n_estimators=800, max_depth=15),
        "xgboost":       dict(n_estimators=300, learning_rate=0.1,
                              max_depth=6, subsample=0.8, colsample_bytree=0.8),
        "lightgbm":      dict(n_estimators=300, learning_rate=0.1),
    }

    @staticmethod
    def get(
        name: str,
        random_state: int = 42,
        # ── search strategy ──
        search_strategy: str = "manual",   # 'manual' | 'grid' | 'random' | 'bayesian'
        # ── cross-validation ──
        cv_strategy: str = "stratified_kfold",  # 'train_test' | 'kfold' | 'stratified_kfold'
        n_splits: int = 5,
        # ── tuning budget ──
        n_iter: int = 20,      # for random search
        n_trials: int = 30,    # for bayesian / optuna
        # ── data (required for grid / random / bayesian) ──
        X_train=None,
        y_train=None,
        # ── scoring ──
        scoring: str = "f1_macro",
        verbose: int = 1,
    ):
        name_lower = name.lower()

        # ── 1. MANUAL (no tuning needed) ──────────────────────
        if search_strategy == "manual":
            params = ModelFactory._MANUAL_PARAMS.get(name_lower, {})
            print(f"[Manual] {name} | params: {params}")
            return _build_base(name_lower, random_state, **params)

        # For all other strategies we need training data
        if X_train is None or y_train is None:
            raise ValueError(
                "X_train and y_train must be provided for "
                f"search_strategy='{search_strategy}'"
            )

        # ── Build CV splitter ──────────────────────────────────
        cv = ModelFactory._make_cv(
            cv_strategy, n_splits, random_state
        )

        # ── 2. GRID SEARCH ────────────────────────────────────
        if search_strategy == "grid":
            base  = _build_base(name_lower, random_state)
            grid  = PARAM_GRIDS.get(name_lower, {})
            print(f"[GridSearch] {name} | cv={cv_strategy}({n_splits}) "
                  f"| total combos={np.prod([len(v) for v in grid.values()])}")
            gs = GridSearchCV(
                base, grid,
                cv=cv, scoring=scoring,
                n_jobs=-1, verbose=verbose
            )
            gs.fit(X_train, y_train)
            print(f"   Best params : {gs.best_params_}")
            print(f"   Best {scoring}: {gs.best_score_:.4f}")
            return gs.best_estimator_

        # ── 3. RANDOM SEARCH ──────────────────────────────────
        if search_strategy == "random":
            base = _build_base(name_lower, random_state)
            grid = PARAM_GRIDS.get(name_lower, {})
            print(f"[RandomSearch] {name} | cv={cv_strategy}({n_splits}) "
                  f"| n_iter={n_iter}")
            rs = RandomizedSearchCV(
                base, grid,
                n_iter=n_iter, cv=cv,
                scoring=scoring, random_state=random_state,
                n_jobs=-1, verbose=verbose
            )
            rs.fit(X_train, y_train)
            print(f"   Best params : {rs.best_params_}")
            print(f"   Best {scoring}: {rs.best_score_:.4f}")
            return rs.best_estimator_

        # ── 4. BAYESIAN (Optuna TPE) ──────────────────────────
        if search_strategy == "bayesian":
            print(f"[Bayesian/Optuna] {name} | cv={cv_strategy}({n_splits}) "
                  f"| n_trials={n_trials}")

            def objective(trial):
                params = _suggest_params(trial, name_lower)
                model  = _build_base(name_lower, random_state, **params)
                scores = cross_val_score(
                    model, X_train, y_train,
                    cv=cv, scoring=scoring, n_jobs=-1
                )
                return scores.mean()

            study = optuna.create_study(
                direction="maximize",
                sampler=optuna.samplers.TPESampler(seed=random_state)
            )
            study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

            best = study.best_params
            print(f"   Best params : {best}")
            print(f"   Best {scoring}: {study.best_value:.4f}")
            return _build_base(name_lower, random_state, **best)

        raise ValueError(
            f"Unknown search_strategy: '{search_strategy}'. "
            "Choose from: manual | grid | random | bayesian"
        )

    # ── CV factory helper ──────────────────────────────────────
    @staticmethod
    def _make_cv(strategy, n_splits, random_state):
        strategy = strategy.lower()
        if strategy == "train_test":
            # sklearn CV=int means StratifiedKFold-2; we signal 'hold-out' via n_splits=2
            print("  [CV] Simple train/test split (n_splits=2 fold approximation)")
            return StratifiedKFold(n_splits=2, shuffle=True, random_state=random_state)
        elif strategy == "kfold":
            print(f"  [CV] K-Fold (k={n_splits})")
            return KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        elif strategy == "stratified_kfold":
            print(f"  [CV] Stratified K-Fold (k={n_splits})")
            return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        else:
            raise ValueError(
                f"Unknown cv_strategy: '{strategy}'. "
                "Choose from: train_test | kfold | stratified_kfold"
            )


### 5. Cross-Validation Evaluator

> A standalone utility to evaluate any fitted (or unfitted) model using
> Train/Test Split, K-Fold, or Stratified K-Fold cross-validation.


In [13]:
from sklearn.model_selection import (
    cross_validate, StratifiedKFold, KFold, train_test_split
)
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score
)
import numpy as np


class CrossValidator:
    """
    Evaluates a model with three CV strategies:

    strategy       k / split   Notes
    ------------   ---------   -------------------------------------------
    train_test     test_size   Fast; high variance; no repeated eval
    kfold          n_splits    Standard K-Fold (may mis-balance classes)
    stratified_kfold n_splits  Preserves class distribution — recommended
                               for classification tasks
    """

    def __init__(
        self,
        strategy: str = "stratified_kfold",
        n_splits: int = 5,
        test_size: float = 0.2,
        random_state: int = 42,
        scoring: list = None,
    ):
        self.strategy     = strategy.lower()
        self.n_splits     = n_splits
        self.test_size    = test_size
        self.random_state = random_state
        self.scoring      = scoring or ["accuracy", "f1_macro",
                                        "precision_macro", "recall_macro"]

    # ──────────────────────────────────────────────────────────
    def evaluate(self, model, X, y) -> dict:
        """Returns a dict with mean ± std for every requested metric."""

        if self.strategy == "train_test":
            return self._train_test_eval(model, X, y)

        cv = self._make_splitter()
        results = cross_validate(
            model, X, y,
            cv=cv,
            scoring=self.scoring,
            n_jobs=-1,
            return_train_score=False
        )

        summary = {}
        for metric in self.scoring:
            scores = results[f"test_{metric}"]
            summary[metric] = {
                "mean":  round(scores.mean(), 4),
                "std":   round(scores.std(),  4),
                "scores": scores.tolist(),
            }

        self._print_summary(summary)
        return summary

    # ──────────────────────────────────────────────────────────
    def _train_test_eval(self, model, X, y) -> dict:
        """Single hold-out split evaluation."""
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y,
            test_size=self.test_size,
            random_state=self.random_state,
            stratify=y
        )
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)

        summary = {
            "accuracy":         {"mean": round(accuracy_score(y_te, y_pred), 4)},
            "f1_macro":         {"mean": round(f1_score(y_te, y_pred, average="macro", zero_division=0), 4)},
            "precision_macro":  {"mean": round(precision_score(y_te, y_pred, average="macro", zero_division=0), 4)},
            "recall_macro":     {"mean": round(recall_score(y_te, y_pred, average="macro", zero_division=0), 4)},
        }
        print(f"\n[Train/Test Split  test_size={self.test_size}]")
        self._print_summary(summary)
        return summary

    # ──────────────────────────────────────────────────────────
    def _make_splitter(self):
        if self.strategy == "kfold":
            print(f"\n[K-Fold  k={self.n_splits}]")
            return KFold(
                n_splits=self.n_splits,
                shuffle=True,
                random_state=self.random_state
            )
        elif self.strategy == "stratified_kfold":
            print(f"\n[Stratified K-Fold  k={self.n_splits}]")
            return StratifiedKFold(
                n_splits=self.n_splits,
                shuffle=True,
                random_state=self.random_state
            )
        else:
            raise ValueError(f"Unknown strategy: {self.strategy}")

    # ──────────────────────────────────────────────────────────
    @staticmethod
    def _print_summary(summary: dict):
        for metric, vals in summary.items():
            std_str = f" ± {vals['std']:.4f}" if "std" in vals else ""
            print(f"   {metric:<22} {vals['mean']:.4f}{std_str}")


### 6. Refactored Trainer

In [14]:
import os
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
from sklearn.preprocessing import LabelEncoder


class Trainer:
    """
    Orchestrates the full training pipeline:
        load → encode → split → vectorize → (optionally resample)
        → ModelFactory (with HP tuning & CV) → evaluate → MLflow log

    search_strategy : 'manual' | 'grid' | 'random' | 'bayesian'
    cv_strategy     : 'train_test' | 'kfold' | 'stratified_kfold'
    """

    def __init__(
        self,
        experiment_name="Final Experiment using HyperParameter Tuning",
        text_column="clean_comment",
        target_column="category",
        test_size=0.2,
        random_state=42
    ):
        self.text_column   = text_column
        self.target_column = target_column
        self.test_size     = test_size
        self.random_state  = random_state
        self.label_encoder = LabelEncoder()

        mlflow.set_tracking_uri("https://mlflow-dashboard.duckdns.org")
        mlflow.set_experiment(experiment_name)

    # ======================
    # Label Encoding
    # ======================
    def _encode_labels(self, y):
        return self.label_encoder.fit_transform(y)

    # ======================
    # Feature Casting
    # ======================
    def _cast_features(self, X, model_name):
        if model_name.lower() in ["lightgbm", "xgboost"]:
            return X.astype("float32")
        return X

    # ======================
    # Run Experiment
    # ======================
    def train(
        self,
        df,
        vectorizer_name="tfidf",
        model_name="random_forest",
        imbalance_method="class_weights",
        ngram_range=(1, 3),
        max_features=10000,
        # ── Hyperparameter tuning ──
        search_strategy: str = "manual",          # 'manual' | 'grid' | 'random' | 'bayesian'
        cv_strategy: str     = "stratified_kfold",# 'train_test' | 'kfold' | 'stratified_kfold'
        n_splits: int        = 5,
        n_iter: int          = 20,    # for random search
        n_trials: int        = 30,    # for bayesian search
        scoring: str         = "f1_macro",
    ):
        # ======================
        # Clean data
        # ======================
        df = df.copy()
        df[self.text_column]   = df[self.text_column].fillna("")
        df[self.target_column] = df[self.target_column].fillna("unknown")

        X = df[self.text_column]
        y = self._encode_labels(df[self.target_column])

        # ======================
        # Split
        # ======================
        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=self.test_size,
            random_state=self.random_state,
            stratify=y
        )

        # ======================
        # Build vectorizer
        # ======================
        vectorizer = VectorizerFactory.get(
            vectorizer_name,
            ngram_range=ngram_range,
            max_features=max_features
        )

        # ======================
        # Vectorize
        # ======================
        X_train_vec = vectorizer.fit_transform(X_train)
        X_test_vec  = vectorizer.transform(X_test)

        # ======================
        # Resampling strategy
        # ======================
        if imbalance_method == "class_weights":
            # Applied directly via ModelFactory later if the model supports it
            pass
        else:
            sampler = ImbalanceHandlerFactory.get(
                imbalance_method,
                random_state=self.random_state
            )
            if sampler:
                X_train_vec, y_train = sampler.fit_resample(X_train_vec, y_train)

        # ======================
        # Casting
        # ======================
        X_train_vec = self._cast_features(X_train_vec, model_name)
        X_test_vec  = self._cast_features(X_test_vec,  model_name)

        # ======================
        # Build / Tune model
        # via ModelFactory (HP search + CV done here)
        # ======================
        model = ModelFactory.get(
            name             = model_name,
            random_state     = self.random_state,
            search_strategy  = search_strategy,
            cv_strategy      = cv_strategy,
            n_splits         = n_splits,
            n_iter           = n_iter,
            n_trials         = n_trials,
            X_train          = X_train_vec,
            y_train          = y_train,
            scoring          = scoring,
            verbose          = 0,
        )

        # Apply class_weight after tuning if relevant
        if imbalance_method == "class_weights" and hasattr(model, "class_weight"):
            model.set_params(class_weight="balanced")

        # ======================
        # MLflow
        # ======================
        with mlflow.start_run():

            # ── Log params ──────────────────────────────────
            mlflow.log_param("vectorizer",       vectorizer_name)
            mlflow.log_param("model",            model_name)
            mlflow.log_param("imbalance_method", imbalance_method)
            mlflow.log_param("ngram_range",      str(ngram_range))
            mlflow.log_param("max_features",     max_features)
            mlflow.log_param("search_strategy",  search_strategy)
            mlflow.log_param("cv_strategy",      cv_strategy)
            mlflow.log_param("n_splits",         n_splits)

            # Log best hyperparams found (if model exposes get_params)
            if hasattr(model, "get_params"):
                for k, v in model.get_params().items():
                    try:
                        mlflow.log_param(f"hp_{k}", v)
                    except Exception:
                        pass

            # ── Train on full training set ───────────────────
            model.fit(X_train_vec, y_train)

            # ── Predict ──────────────────────────────────────
            y_pred = model.predict(X_test_vec)

            # ── Metrics ──────────────────────────────────────
            acc    = accuracy_score(y_test, y_pred)
            report = classification_report(y_test, y_pred, output_dict=True)

            mlflow.log_metric("accuracy",         acc)
            mlflow.log_metric("f1_macro",         report["macro avg"]["f1-score"])
            mlflow.log_metric("precision_macro",  report["macro avg"]["precision"])
            mlflow.log_metric("recall_macro",     report["macro avg"]["recall"])

            # ── Confusion Matrix ─────────────────────────────
            cm = confusion_matrix(y_test, y_pred)
            plt.figure(figsize=(8, 6))
            sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
            plt.title(f"{model_name} | {search_strategy} | {imbalance_method}")
            os.makedirs("artifacts", exist_ok=True)
            cm_path = f"artifacts/cm_{model_name}_{search_strategy}.png"
            plt.savefig(cm_path)
            plt.close()
            mlflow.log_artifact(cm_path)

            # ── Log model ────────────────────────────────────
            mlflow.sklearn.log_model(model, artifact_path="model")

            print(f"\nAccuracy: {acc:.4f}")
            print(classification_report(y_test, y_pred))

        return model, vectorizer


### 7. Usage Examples

In [16]:
# ── Standalone CrossValidator ──
from scipy.sparse import csr_matrix
# After running trainer.train(), it returns (model, vectorizer)
model, vectorizer = trainer.train(
    df,
    model_name='xgboost',
    search_strategy='manual'
)

# Vectorize the full dataset
X_vec = vectorizer.transform(df['clean_comment']).astype('float32')
y     = LabelEncoder().fit_transform(df['category'])

# Now evaluate
cv_eval = CrossValidator(strategy='stratified_kfold', n_splits=5)
cv_eval.evaluate(model, X_vec, y)

[Manual] xgboost | params: {'n_estimators': 300, 'learning_rate': 0.1, 'max_depth': 6, 'subsample': 0.8, 'colsample_bytree': 0.8}


2026/05/09 21:45:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/09 21:45:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Accuracy: 0.8047
              precision    recall  f1-score   support

           0       0.75      0.96      0.84      2529
           1       0.85      0.81      0.83      3154
           2       0.84      0.55      0.67      1650

    accuracy                           0.80      7333
   macro avg       0.81      0.77      0.78      7333
weighted avg       0.81      0.80      0.80      7333

🏃 View run upbeat-moth-225 at: https://mlflow-dashboard.duckdns.org/#/experiments/11/runs/c2f0c9c10c80471f96d10eca712c98e0
🧪 View experiment at: https://mlflow-dashboard.duckdns.org/#/experiments/11

[Stratified K-Fold  k=5]
   accuracy               0.8107 ± 0.0032
   f1_macro               0.7887 ± 0.0049
   precision_macro        0.8188 ± 0.0039
   recall_macro           0.7831 ± 0.0048


{'accuracy': {'mean': np.float64(0.8107),
  'std': np.float64(0.0032),
  'scores': [0.808673121505523,
   0.8074457929905905,
   0.8151936715766503,
   0.8082378614293508,
   0.8138297872340425]},
 'f1_macro': {'mean': np.float64(0.7887),
  'std': np.float64(0.0049),
  'scores': [0.7856120766921996,
   0.7837620812790536,
   0.79440447602326,
   0.7850346762495142,
   0.7948857060604709]},
 'precision_macro': {'mean': np.float64(0.8188),
  'std': np.float64(0.0039),
  'scores': [0.8166678564487299,
   0.8172360052321604,
   0.8233149049723929,
   0.8135175239838758,
   0.8233090558929455]},
 'recall_macro': {'mean': np.float64(0.7831),
  'std': np.float64(0.0048),
  'scores': [0.7798498668305974,
   0.778082222620819,
   0.7889696922874102,
   0.7797268122139908,
   0.7888646734292887]}}

In [17]:
# ── Standalone CrossValidator (uses model + vectorizer from above) ────────
from sklearn.preprocessing import LabelEncoder

# Vectorize the full dataset with the same fitted vectorizer
X_vec = vectorizer.transform(df['clean_comment']).astype('float32')
y     = LabelEncoder().fit_transform(df['category'])

cv_eval = CrossValidator(strategy='stratified_kfold', n_splits=5)
cv_eval.evaluate(model, X_vec, y)


[Stratified K-Fold  k=5]
   accuracy               0.8107 ± 0.0032
   f1_macro               0.7887 ± 0.0049
   precision_macro        0.8188 ± 0.0039
   recall_macro           0.7831 ± 0.0048


{'accuracy': {'mean': np.float64(0.8107),
  'std': np.float64(0.0032),
  'scores': [0.808673121505523,
   0.8074457929905905,
   0.8151936715766503,
   0.8082378614293508,
   0.8138297872340425]},
 'f1_macro': {'mean': np.float64(0.7887),
  'std': np.float64(0.0049),
  'scores': [0.7856120766921996,
   0.7837620812790536,
   0.79440447602326,
   0.7850346762495142,
   0.7948857060604709]},
 'precision_macro': {'mean': np.float64(0.8188),
  'std': np.float64(0.0039),
  'scores': [0.8166678564487299,
   0.8172360052321604,
   0.8233149049723929,
   0.8135175239838758,
   0.8233090558929455]},
 'recall_macro': {'mean': np.float64(0.7831),
  'std': np.float64(0.0048),
  'scores': [0.7798498668305974,
   0.778082222620819,
   0.7889696922874102,
   0.7797268122139908,
   0.7888646734292887]}}

In [18]:
# ── 2. Grid Search + Stratified K-Fold ───────────────────────────────────
model_grid, vectorizer_grid = trainer.train(
    df,
    model_name='random_forest',
    search_strategy='grid',
    cv_strategy='stratified_kfold',
    n_splits=5
)

  [CV] Stratified K-Fold (k=5)
[GridSearch] random_forest | cv=stratified_kfold(5) | total combos=72
   Best params : {'max_depth': None, 'max_features': 'sqrt', 'min_samples_split': 10, 'n_estimators': 100}
   Best f1_macro: 0.7499


2026/05/09 23:56:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/09 23:56:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Accuracy: 0.8162
              precision    recall  f1-score   support

           0       0.84      0.93      0.88      2529
           1       0.79      0.88      0.83      3154
           2       0.83      0.53      0.64      1650

    accuracy                           0.82      7333
   macro avg       0.82      0.78      0.79      7333
weighted avg       0.82      0.82      0.81      7333

🏃 View run abrasive-pig-778 at: https://mlflow-dashboard.duckdns.org/#/experiments/11/runs/3ba3e0081f0a4af3afb0475b32d5b3e0
🧪 View experiment at: https://mlflow-dashboard.duckdns.org/#/experiments/11


In [19]:
# ── 3. Random Search + K-Fold ─────────────────────────────────────────────
model_random, vectorizer_random = trainer.train(
    df,
    model_name='lightgbm',
    search_strategy='random',
    cv_strategy='kfold',
    n_splits=5,
    n_iter=20
)

  [CV] K-Fold (k=5)
[RandomSearch] lightgbm | cv=kfold(5) | n_iter=20
   Best params : {'num_leaves': 63, 'n_estimators': 100, 'max_depth': -1, 'learning_rate': 0.1}
   Best f1_macro: 0.8471


d:\MyFiles\GitHub\YouTube Viewer Sentiment Analysis\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/05/10 05:49:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/10 05:49:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Accuracy: 0.8655
              precision    recall  f1-score   support

           0       0.85      0.96      0.90      2529
           1       0.91      0.84      0.88      3154
           2       0.81      0.76      0.79      1650

    accuracy                           0.87      7333
   macro avg       0.86      0.86      0.85      7333
weighted avg       0.87      0.87      0.86      7333

🏃 View run zealous-cat-644 at: https://mlflow-dashboard.duckdns.org/#/experiments/11/runs/83907988e0cf4963b611c2ef2574b061
🧪 View experiment at: https://mlflow-dashboard.duckdns.org/#/experiments/11


In [20]:
# ── 4. Bayesian Optimization (Optuna) + Stratified K-Fold ─────────────────
model_bayes, vectorizer_bayes = trainer.train(
    df,
    model_name='xgboost',
    search_strategy='bayesian',
    cv_strategy='stratified_kfold',
    n_splits=5,
    n_trials=30
)

  [CV] Stratified K-Fold (k=5)
[Bayesian/Optuna] xgboost | cv=stratified_kfold(5) | n_trials=30


Best trial: 11. Best value: 0.828193: 100%|██████████| 30/30 [4:18:28<00:00, 516.95s/it]  


   Best params : {'n_estimators': 181, 'learning_rate': 0.28031697045299836, 'max_depth': 10, 'subsample': 0.5427925342039944, 'colsample_bytree': 0.5018684889065917}
   Best f1_macro: 0.8282


2026/05/10 10:13:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/10 10:13:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Accuracy: 0.8474
              precision    recall  f1-score   support

           0       0.83      0.95      0.89      2529
           1       0.87      0.86      0.87      3154
           2       0.84      0.66      0.74      1650

    accuracy                           0.85      7333
   macro avg       0.84      0.82      0.83      7333
weighted avg       0.85      0.85      0.84      7333

🏃 View run sincere-boar-213 at: https://mlflow-dashboard.duckdns.org/#/experiments/11/runs/63dae9b28cf3413ebe73ff548bc500ca
🧪 View experiment at: https://mlflow-dashboard.duckdns.org/#/experiments/11


In [ ]:
# ── 3. Bayesian Search (Optuna) + K-Fold ─────────────────────────────────────────────
model_random, vectorizer_random = trainer.train(
    df,
    model_name='lightgbm',
    search_strategy='bayesian',
    cv_strategy='kfold',
    n_splits=5,
    n_iter=20
)